# 02. MONAI Transforms — 의료영상 전처리·증강

**MONAI** 는 PyTorch 기반 의료영상 딥러닝 프레임워크입니다.
의료영상에 특화된 전처리·증강 변환을 제공해, 일관된 파이프라인을 쉽게 구성할 수 있습니다.

1. 합성 볼륨 준비
2. 전처리 변환 (채널 추가, 강도 정규화)
3. 증강 변환 (뒤집기, 회전)
4. Compose로 파이프라인 구성

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from monai.transforms import (
    Compose, EnsureChannelFirst, ScaleIntensity,
    RandFlip, RandRotate90, RandGaussianNoise,
)

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False
import monai
print('MONAI', monai.__version__)

## 1. 합성 볼륨 준비

2D 의료영상(예: 한 장의 슬라이스)을 모사한 합성 데이터를 만듭니다.

In [ ]:
size = 128
yy, xx = np.mgrid[:size, :size]
image = (((xx-64)**2 + (yy-64)**2 < 40**2).astype('float32') * 200
         + 30 * np.random.randn(size, size)).astype('float32')
print('원본 shape:', image.shape, '| 강도 범위:', image.min().round(1), '~', image.max().round(1))

## 2. 전처리 변환

- `EnsureChannelFirst`: 채널 차원을 맨 앞에 추가 (모델 입력 형식)
- `ScaleIntensity`: 강도를 0~1 범위로 정규화

In [ ]:
from monai.transforms import EnsureChannelFirst
img = image[np.newaxis]   # 채널 추가 (1, H, W) — 이미 채널-퍼스트 형태로 준비
scale = ScaleIntensity()
scaled = scale(img)
print('정규화 후 강도 범위:', float(scaled.min()), '~', float(scaled.max()))

## 3. 증강 변환 + Compose

여러 변환을 `Compose`로 묶어 하나의 파이프라인으로 만듭니다. 무작위 증강은 매번 다른 결과를 만들어 데이터 다양성을 높입니다.

In [ ]:
train_transforms = Compose([
    ScaleIntensity(),
    RandFlip(prob=0.5, spatial_axis=1),
    RandRotate90(prob=0.5),
    RandGaussianNoise(prob=0.5, std=0.05),
])

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(image, cmap='gray'); axes[0].set_title('원본'); axes[0].axis('off')
for i in range(1, 4):
    aug = train_transforms(img)
    axes[i].imshow(np.asarray(aug)[0], cmap='gray')
    axes[i].set_title(f'증강 {i}'); axes[i].axis('off')
plt.suptitle('MONAI 변환 파이프라인', fontsize=13)
plt.tight_layout(); plt.show()

### 정리

- MONAI transforms로 의료영상의 채널 정리, 강도 정규화, 무작위 증강을 적용했습니다.
- `Compose`로 여러 변환을 하나의 파이프라인으로 묶었습니다.
- 이 전처리는 다음 노트북의 분할 모델 학습에 그대로 쓰입니다.